# Airfare insights notebook

This notebook builds summary charts from `dm.dm_route_purchase_recommendations`.

In [5]:
import os

import pandas as pd
import plotly.express as px
import psycopg2


def _clean(value: str | None, default: str) -> str:
    raw = (value or default).strip()
    # Remove accidental NULL bytes copied from env files/editors.
    return raw.replace("\x00", "")


user = _clean(os.getenv("POSTGRES_USER"), "dwh")
password = _clean(os.getenv("POSTGRES_PASSWORD"), "dwh")
host = _clean(os.getenv("POSTGRES_HOST"), "localhost")
port = int(_clean(os.getenv("POSTGRES_PORT"), "5432"))
db = _clean(os.getenv("POSTGRES_DB"), "dwh")


def connect_postgres():
    # Try env values first, then common local defaults.
    candidates = [
        (host, port, db, user, password),
        ("localhost", 5432, "dwh", "dwh", "dwh"),
        ("localhost", 5432, "dwh", "dwh", "change_me"),
        ("127.0.0.1", 5432, "dwh", "dwh", "dwh"),
        ("127.0.0.1", 5432, "dwh", "dwh", "change_me"),
    ]

    unique_candidates = []
    seen = set()
    for item in candidates:
        if item not in seen:
            unique_candidates.append(item)
            seen.add(item)

    last_error = None
    for h, p, d, u, pwd in unique_candidates:
        try:
            conn = psycopg2.connect(
                host=h,
                port=p,
                dbname=d,
                user=u,
                password=pwd,
            )
            print(f"Connected to Postgres: host={h} db={d} user={u}")
            return conn
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            continue

    raise RuntimeError(f"Could not connect to Postgres. Last error: {last_error}")


def safe_show(fig):
    """Render Plotly figure inline; fallback to browser if notebook renderer is unavailable."""
    try:
        fig.show()
    except ValueError as exc:
        if "nbformat" in str(exc).lower() or "mime type rendering" in str(exc).lower():
            fig.show(renderer="browser")
        else:
            raise

In [6]:
queries = [
    """
    select
        route_origin,
        route_destination,
        best_days_before_departure,
        best_search_weekday,
        best_search_hour,
        route_avg_price,
        route_price_stddev,
        route_price_cv,
        route_min_price,
        route_max_price
    from dm.dm_route_purchase_recommendations
    """,
    """
    select
        route_origin,
        route_destination,
        best_days_before_departure,
        best_search_weekday,
        best_search_hour,
        route_avg_price,
        route_price_stddev,
        route_price_cv,
        route_min_price,
        route_max_price
    from analytics_dm.dm_route_purchase_recommendations
    """,
]

conn = connect_postgres()
try:
    last_error = None
    for query in queries:
        try:
            df = pd.read_sql_query(query, conn)
            break
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            continue
    else:
        raise RuntimeError(f"Could not read DM table. Last error: {last_error}")
finally:
    conn.close()

print(f"Rows loaded: {len(df)}")
df.head()

Connected to Postgres: host=localhost db=dwh user=dwh
Rows loaded: 9


C:\Users\saifu\AppData\Local\Temp\ipykernel_25624\1944451664.py:37: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



,route_origin,route_destination,best_days_before_departure,best_search_weekday,best_search_hour,route_avg_price,route_price_stddev,route_price_cv,route_min_price,route_max_price
0,DME,LED,57,Monday,0,18033.36,5230.74,0.2901,7100.13,31086.99
1,SVO,LED,60,Monday,0,13066.60,3962.47,0.3033,8077.73,31709.14
2,VKO,IST,58,Sunday,17,10594.99,4126.24,0.3895,6361.41,30699.93
3,DME,DXB,58,Monday,0,20480.25,4472.70,0.2184,11693.21,30307.51
4,VKO,LED,57,Sunday,17,11825.18,3038.58,0.2570,6952.96,24864.91


In [7]:
route_avg = df.sort_values("route_avg_price")
fig_avg = px.bar(
    route_avg,
    x="route_origin",
    y="route_avg_price",
    color="route_destination",
    title="Average airfare by route",
)
safe_show(fig_avg)

In [8]:
fig_volatility = px.scatter(
    df,
    x="route_avg_price",
    y="route_price_stddev",
    color="route_destination",
    hover_data=[
        "route_origin",
        "best_days_before_departure",
        "best_search_weekday",
        "best_search_hour",
        "route_price_cv",
    ],
    title="Price level vs volatility by route",
)
safe_show(fig_volatility)

df[
    [
        "route_origin",
        "route_destination",
        "best_days_before_departure",
        "best_search_weekday",
        "best_search_hour",
        "route_avg_price",
        "route_price_stddev",
        "route_price_cv",
    ]
].sort_values("route_avg_price").head(20)

,route_origin,route_destination,best_days_before_departure,best_search_weekday,best_search_hour,route_avg_price,route_price_stddev,route_price_cv
2,VKO,IST,58,Sunday,17,10594.99,4126.24,0.3895
5,SVO,IST,58,Tuesday,13,11186.58,2767.03,0.2474
4,VKO,LED,57,Sunday,17,11825.18,3038.58,0.2570
1,SVO,LED,60,Monday,0,13066.60,3962.47,0.3033
6,SVO,DXB,58,Sunday,17,14410.89,5124.42,0.3556
8,VKO,DXB,58,Tuesday,13,16259.52,3825.87,0.2353
0,DME,LED,57,Monday,0,18033.36,5230.74,0.2901
3,DME,DXB,58,Monday,0,20480.25,4472.70,0.2184
7,DME,IST,51,Tuesday,13,21589.79,5721.48,0.2650
